# Using CO2 PPM to Predict Extreme Weather Pattens

**Team members:** [Sherlock Chen, Edward Christopher, Henry Fong]  
**Repository:** [https://github.com/cjulianan/DS_3001_Semester_Project]

## Setup

Use relative paths and keep the notebook runnable from beginning to end.

In [1]:
from pathlib import Path

DATA_DIR = Path("data")
assert DATA_DIR.is_dir(), "The data directory is missing."

## Import library

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay

## Load data

Sources Used:

https://www.geeksforgeeks.org/python/how-to-skip-rows-while-reading-csv-file-using-pandas/

https://stackoverflow.com/questions/18366797/pandas-read-csv-how-to-skip-comment-lines


In [3]:
# Load data from DATA_DIR.

CO2_data = pd.read_excel("data/CO2 PPM over Time.xlsx") # Loads CO2 data
Weather_data = pd.read_excel("data/public_emdat_custom_request.xlsx") # Loads weather data
CH4_data = pd.read_csv("data/ch4_mm_gl.csv", comment = '#') # Loads CH4 data
Nina4_data = pd.read_csv("data/nina4.anom.csv") # Loads nina4 data
Global_temp_data = pd.read_csv("data/global_temp_anomaly.csv", skiprows=1) # Loads Global temp anomaly data
DMI_data = pd.read_csv("data/dmi.had.long.csv") # Loads dmi data

## Prepare data

Sources Used:

https://www.geeksforgeeks.org/python/get-the-data-type-of-column-in-pandas-python/

https://www.geeksforgeeks.org/pandas/python-pandas-dataframe-describe-method/

https://www.geeksforgeeks.org/pandas/how-to-rename-columns-in-pandas-dataframe/

https://www.geeksforgeeks.org/python/filter-pandas-dataframe-by-time/

https://www.geeksforgeeks.org/python/pandas-drop-column/

https://www.geeksforgeeks.org/pandas/python-pandas-dataframe-drop_duplicates/

https://www.geeksforgeeks.org/python/python-pandas-dataframe-dropna/

https://www.geeksforgeeks.org/data-analysis/python-pandas-dataframe-replace/

https://www.geeksforgeeks.org/pandas/python-pandas-split-strings-into-two-list-columns-using-str-split/

https://www.geeksforgeeks.org/python/split-a-text-column-into-two-columns-in-pandas-dataframe/

https://www.geeksforgeeks.org/python/how-to-convert-string-to-integer-in-pandas-dataframe/

## CO2 PPM Data

In [4]:
# Overview of CO2 Data
CO2_datatype = CO2_data.dtypes
display(CO2_datatype)
display(CO2_data.head(5))
display(CO2_data.describe())

,0
year,int64
month,int64
decimal date,float64
monthly average,float64
de-season alized,float64
#days,int64
st.dev of days,float64
unc. of mon mean,float64


,year,month,decimal date,monthly average,de-season alized,#days,st.dev of days,unc. of mon mean
0,1958,3,1958.2027,315.71,314.44,-1,-9.99,-0.99
1,1958,4,1958.2877,317.45,315.16,-1,-9.99,-0.99
2,1958,5,1958.3699,317.51,314.69,-1,-9.99,-0.99
3,1958,6,1958.4548,317.27,315.15,-1,-9.99,-0.99
4,1958,7,1958.5370,315.87,315.20,-1,-9.99,-0.99


,year,month,decimal date,monthly average,de-season alized,#days,st.dev of days,unc. of mon mean
count,823.000000,823.000000,823.000000,823.000000,823.000000,823.000000,823.000000,823.000000
mean,1991.958688,6.495747,1992.457736,361.439101,361.431786,19.165249,-1.991592,-0.084593
std,19.812337,3.444685,19.811122,33.511832,33.451725,11.810524,4.478122,0.508127
min,1958.000000,1.000000,1958.202700,312.420000,314.440000,-1.000000,-9.990000,-0.990000
25%,1975.000000,4.000000,1975.333350,331.285000,330.915000,14.500000,0.210000,0.080000
50%,1992.000000,6.000000,1992.458300,356.380000,356.500000,25.000000,0.400000,0.150000
75%,2009.000000,9.000000,2009.583350,387.845000,387.825000,28.000000,0.570000,0.220000
max,2026.000000,12.000000,2026.708300,432.340000,429.550000,31.000000,1.310000,0.580000


In [5]:
# Cleaned CO2 PPM of any null data or duplicates
CO2_data.drop_duplicates(inplace=True)
CO2_data.dropna(inplace=True)

In [6]:
# Removed unnecessary data
"""
Dropped decimal date, #days, st.dev of days, and unc. of mon mean as they are days or uncertainty factor that does not play a major role in the data analysis
Dropped monthly average but not de-seasonalized as it is monthly average however it removed seasonal influences such as crop growth during the summer
"""
CO2_data_dropped = CO2_data.drop(['decimal date', 'monthly average', '#days', 'st.dev of days', 'unc. of mon mean'], axis = 1)
Date1 = CO2_data['year'] >= 2000
Date2 = CO2_data['year'] <= 2025
CO2_data_cleaned = CO2_data_dropped.loc[Date1 & Date2]
CO2_data_cleaned.columns = ['Year', 'Month', 'De-seasonalized'] # Renamed column to start with a uppercase letter for data consistency
display(CO2_data_cleaned.head(5))

,Year,Month,De-seasonalized
502,2000,1,369.24
503,2000,2,368.99
504,2000,3,369.24
505,2000,4,369.44
506,2000,5,368.87


## CH4 Data

In [7]:
# Overview of CH4 Data
CH4_datatype = CH4_data.dtypes
display(CH4_datatype)
display(CH4_data.head(5))
display(CH4_data.describe())

,0
year,int64
month,int64
decimal,float64
average,float64
average_unc,float64
trend,float64
trend_unc,float64


,year,month,decimal,average,average_unc,trend,trend_unc
0,1983,7,1983.542,1625.96,2.25,1635.08,1.50
1,1983,8,1983.625,1628.04,2.83,1635.58,1.44
2,1983,9,1983.708,1638.42,2.45,1636.15,1.36
3,1983,10,1983.792,1644.81,1.58,1636.78,1.27
4,1983,11,1983.875,1642.60,0.79,1637.49,1.18


,year,month,decimal,average,average_unc,trend,trend_unc
count,518.000000,518.000000,518.000000,518.000000,518.000000,518.000000,518.000000
mean,2004.583012,6.503861,2005.083334,1787.537124,0.901544,1787.576718,0.461042
std,12.479301,3.449411,12.473165,75.589171,1.303039,75.522215,1.230411
min,1983.000000,1.000000,1983.542000,1625.960000,-9.990000,1635.080000,-9.990000
25%,1994.000000,4.000000,1994.312750,1741.520000,0.870000,1741.192500,0.530000
50%,2005.000000,7.000000,2005.083500,1777.440000,1.000000,1775.365000,0.590000
75%,2015.000000,9.000000,2015.854250,1836.675000,1.180000,1837.927500,0.647500
max,2026.000000,12.000000,2026.625000,1945.470000,2.830000,1943.590000,1.500000


In [8]:
# Cleaned CH4 of any null data or duplicates
CH4_data.drop_duplicates(inplace=True)
CH4_data.dropna(inplace=True)

In [9]:
# Removed unnecessary data
"""
Removed decimal, average_unc, trend_unc as decimal is a time data and trend_unc and average_unc track uncertainty which does not play a major role in data analysis
Used trend instead of average as trend removed the seasonal impact that occur every year helping to smooth out the data points
"""
CH4_data_dropped = CH4_data.drop(['decimal', 'average', 'average_unc', 'trend_unc'], axis = 1)
Date1 = CH4_data['year'] >= 2000
Date2 = CH4_data['year'] <= 2025
CH4_data_cleaned = CH4_data_dropped.loc[Date1 & Date2]
CH4_data_cleaned.columns = ['Year', 'Month', 'Trend'] # Renamed column to start with a uppercase letter for data consistency
display(CH4_data_cleaned.head(5))

,Year,Month,Trend
198,2000,1,1773.30
199,2000,2,1773.30
200,2000,3,1773.28
201,2000,4,1773.24
202,2000,5,1773.17


## Nina4 data

In [10]:
# Overview of Nina4 Data
Nina4_datatype = Nina4_data.dtypes
display(Nina4_datatype)
display(Nina4_data.head(5))
display(Nina4_data.describe())

,0
Date,object
NINA4 missing value -99.99 https://psl.noaa.gov/data/timeseries/month/,float64


,Date,NINA4 missing value -99.99 https://psl.noaa.gov/data/timeseries/month/
0,1948-01-01,-0.50
1,1948-02-01,-0.43
2,1948-03-01,-0.13
3,1948-04-01,-0.01
4,1948-05-01,0.60


,NINA4 missing value -99.99 https://psl.noaa.gov/data/timeseries/month/
count,948.000000
mean,-42.269705
std,648.470369
min,-9999.000000
25%,-0.530000
50%,-0.050000
75%,0.370000
max,1.580000


In [11]:
# Cleaned Nina4 of any null data or duplicates
Nina4_data.drop_duplicates(inplace=True)
Nina4_data = Nina4_data.replace(to_replace = -9999, value = np.nan)
Nina4_data.dropna(inplace=True)
display(Nina4_data.describe())

,NINA4 missing value -99.99 https://psl.noaa.gov/data/timeseries/month/
count,944.000000
mean,-0.080169
std,0.646837
min,-2.110000
25%,-0.530000
50%,-0.050000
75%,0.370000
max,1.580000


In [12]:
# Filtered data by year and split date into year and month
Nina4_data[['Year', 'Month', 'Day']] = Nina4_data.Date.str.split("-", n=3, expand=True) # Splits Date into Year, Month, Day

# Convert both month and year into int to prevent wrong data type conversion
Nina4_data['Year'] = Nina4_data['Year'].astype(int)
Nina4_data['Month'] = Nina4_data['Month'].astype(int)

Nina4_data_dropped = Nina4_data.drop(['Date', 'Day'], axis = 1)
Date1 = Nina4_data['Year'] >= 2000
Date2 = Nina4_data['Year'] <= 2025
Nina4_data_cleaned = Nina4_data_dropped.loc[Date1 & Date2]
Nina4_data_cleaned.columns = ['Nina4', 'Year', 'Month'] # Renamed column to start with a uppercase letter for data consistency
display(Nina4_data_cleaned.head(5))

,Nina4,Year,Month
624,-1.16,2000,1
625,-1.19,2000,2
626,-1.16,2000,3
627,-1.01,2000,4
628,-0.96,2000,5


## Global temp data

In [13]:
# Overview of Global temp Data
Global_temp_data_datatype = Global_temp_data.dtypes
display(Global_temp_data_datatype)
display(Global_temp_data.head(5))
display(Global_temp_data.describe())

,0
Year,int64
Jan,float64
Feb,float64
Mar,float64
Apr,float64
May,float64
Jun,float64
Jul,float64
Aug,float64
Sep,float64


,Year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,J-D,D-N,DJF,MAM,JJA,SON
0,1880,-0.19,-0.25,-0.09,-0.16,-0.10,-0.21,-0.19,-0.10,-0.15,-.23,-.22,-.18,-.17,***,***,-0.12,-0.17,-.20
1,1881,-0.20,-0.14,0.03,0.05,0.06,-0.19,0.00,-0.03,-0.15,-.22,-.19,-.07,-.09,-.10,-.17,0.05,-0.07,-.18
2,1882,0.16,0.13,0.04,-0.16,-0.14,-0.22,-0.16,-0.08,-0.15,-.23,-.17,-.36,-.11,-.09,.07,-0.09,-0.16,-.18
3,1883,-0.29,-0.37,-0.12,-0.18,-0.18,-0.07,-0.08,-0.14,-0.22,-.11,-.24,-.11,-.18,-.20,-.34,-0.16,-0.10,-.19
4,1884,-0.13,-0.08,-0.36,-0.40,-0.33,-0.35,-0.30,-0.28,-0.27,-.25,-.33,-.31,-.28,-.27,-.11,-0.36,-0.31,-.28


,Year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,MAM,JJA
count,147.000000,147.000000,147.000000,147.000000,147.000000,147.000000,147.000000,147.000000,147.000000,147.000000,147.000000,147.000000
mean,1953.000000,0.086463,0.094966,0.114014,0.085782,0.074150,0.062381,0.085986,0.087279,0.093605,0.091497,0.078639
std,42.579338,0.450120,0.457016,0.464673,0.425306,0.402473,0.404440,0.387706,0.407566,0.408405,0.426437,0.396384
min,1880.000000,-0.810000,-0.620000,-0.640000,-0.580000,-0.550000,-0.520000,-0.510000,-0.540000,-0.570000,-0.580000,-0.490000
25%,1916.500000,-0.240000,-0.230000,-0.220000,-0.245000,-0.235000,-0.245000,-0.190000,-0.205000,-0.190000,-0.250000,-0.210000
50%,1953.000000,-0.010000,-0.030000,0.030000,0.000000,-0.030000,-0.050000,-0.030000,-0.040000,-0.040000,0.000000,-0.040000
75%,1989.500000,0.330000,0.405000,0.355000,0.310000,0.295000,0.280000,0.315000,0.320000,0.295000,0.320000,0.320000
max,2026.000000,1.370000,1.430000,1.390000,1.320000,1.160000,1.230000,1.240000,1.420000,1.480000,1.290000,1.280000


In [14]:
# Cleaned Global temp of any null data or duplicates
Global_temp_data.drop_duplicates(inplace=True)
Global_temp_data = Global_temp_data.replace(to_replace = '***', value = np.nan)
Global_temp_data.dropna(inplace=True)

In [15]:
# Removed unnecessary data
"""
Removed J-D, D-N, DJF, MAM, JJA, SON as these data are tracking seasonal or yearly metrics which we do not need for our data analysis
"""
Global_temp_data_dropped = Global_temp_data.drop(['J-D', 'D-N', 'DJF', 'MAM', 'JJA', 'SON'], axis = 1)
Date1 = Global_temp_data['Year'] >= 2000
Date2 = Global_temp_data['Year'] <= 2025
Global_temp_data_cleaned = Global_temp_data_dropped.loc[Date1 & Date2]
# Renamed column to start with a uppercase letter for data consistency and replaced Month with numbers to ensure data naming consistency
Global_temp_data_cleaned.columns = ['Year', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']
display(Global_temp_data_cleaned.head(5))

,Year,1,2,3,4,5,6,7,8,9,10,11,12
120,2000,0.25,0.56,0.55,0.56,0.35,0.40,0.39,0.42,0.39,.27,.30,.28
121,2001,0.45,0.44,0.56,0.50,0.57,0.52,0.59,0.47,0.52,.50,.72,.56
122,2002,0.77,0.79,0.88,0.58,0.64,0.53,0.61,0.53,0.64,.54,.59,.44
123,2003,0.75,0.58,0.60,0.55,0.60,0.48,0.58,0.63,0.62,.72,.53,.75
124,2004,0.58,0.73,0.63,0.61,0.37,0.44,0.26,0.44,0.50,.61,.73,.51


## DMI data

In [16]:
# Overview of DMI Data
DMI_datatype = DMI_data.dtypes
display(DMI_datatype)
display(DMI_data.head(5))
display(DMI_data.describe())

,0
Date,object
DMI HadISST1.1 missing value -9999 https://psl.noaa.gov/data/timeseries/month/,float64


,Date,DMI HadISST1.1 missing value -9999 https://psl.noaa.gov/data/timeseries/month/
0,1870-01-01,-0.438
1,1870-02-01,-0.337
2,1870-03-01,0.177
3,1870-04-01,-0.048
4,1870-05-01,-0.480


,DMI HadISST1.1 missing value -9999 https://psl.noaa.gov/data/timeseries/month/
count,1884.000000
mean,-26.767551
std,514.552145
min,-9999.000000
25%,-0.453000
50%,-0.245000
75%,-0.021000
max,1.279000


In [17]:
# Cleaned DMI of any null data or duplicates
DMI_data.drop_duplicates(inplace=True)
DMI_data = DMI_data.replace(to_replace = -9999, value = np.nan)
DMI_data.dropna(inplace=True)
display(DMI_data.describe())

,DMI HadISST1.1 missing value -9999 https://psl.noaa.gov/data/timeseries/month/
count,1879.000000
mean,-0.231542
std,0.346831
min,-1.634000
25%,-0.451500
50%,-0.243000
75%,-0.020500
max,1.279000


In [18]:
# Filtered data by year and split date into year and month
DMI_data[['Year', 'Month', 'Day']] = DMI_data.Date.str.split("-", n=3, expand=True) # Splits Date into Year, Month, Day

# Convert both month and year into int to prevent wrong data type conversion
DMI_data['Year'] = DMI_data['Year'].astype(int)
DMI_data['Month'] = DMI_data['Month'].astype(int)

DMI_data_dropped = DMI_data.drop(['Date', 'Day'], axis = 1)
Date1 = DMI_data['Year'] >= 2000
Date2 = DMI_data['Year'] <= 2025
DMI_data_cleaned = DMI_data_dropped.loc[Date1 & Date2]
DMI_data_cleaned.columns = ['Dmi', 'Year', 'Month'] # Renamed column to start with a uppercase letter for data consistency
display(DMI_data_cleaned.head(5))

,Dmi,Year,Month
1560,-0.125,2000,1
1561,-0.009,2000,2
1562,0.141,2000,3
1563,0.141,2000,4
1564,0.113,2000,5


## Explore data

## Build models

In [19]:
# Train baseline and candidate models.

## Evaluate models

In [20]:
# Compare models using the chosen evaluation measures.

## Final results

In [21]:
# Generate the final tables and figures.

## AI usage disclosure

- **AI tool/model:** [Name the tool and model, or state that no generative AI was used.]
- **Used for:** [Briefly identify the tasks.]
- **How we verified the work:** [Briefly identify the checks.]
- **Prompt(s) used:** [Paste the prompts used.]